In [2]:
from osgeo import gdal
import rasterio
# from rasterio.features import shapes
from rasterio import features
from rasterio.transform import from_origin
import os
from difflib import SequenceMatcher
import numpy as np
import random
import itertools
import pandas as pd
import geopandas as gpd
import re

from collections import defaultdict
from shapely.geometry import Point

In [22]:
def create_folder_if_not_exists(folder_path):
    """
    Create a folder if it doesn't exist.

    Parameters:
    folder_path (str): The path of the folder to be created.
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder created at: {folder_path}")
    else:
        print(f"Folder already exists at: {folder_path}")

def get_vector_file_list(path):
    """
    Get a list of the vector files inside the folder
    Parameters:
    - path (str): path of the folder with the resources.

    Returns:
    - File_list (list). list of the resources.
    """
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        # "anat" is just to get here necessary ones
        if file.endswith(".shp"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def get_raster_file_list(path):
    """
    Get a list of the raster files inside the folder
    Parameters:
    - path (str): path of the folder with the resources.

    Returns:
    - File_list (list). list of the resources.
    """
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        # "32628" is just to get here necessary ones
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def get_csv_file_list(path):
    """
    Get a list of the csv files inside the folder
    Parameters:
    - path (str): path of the folder with the resources.

    Returns:
    - File_list (list). list of the resources.
    """
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".csv"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def update_names_based_on_similarity(unique_names, gdf, column_name, similarity_threshold=0):
    """
    Update names in gdf based on similarity to names in unique list.

    Parameters:
    - unique_names (list): list of the unique names.
    - gdf (GeoDataFrame): GeoDataFrame whose names need to be updated.
    - column_name (str): String of the column.
    - similarity_threshold (float): Threshold for similarity ratio.

    Returns:
    - gdf. Updates gdf in place.
    """
    # Iterate through rows of gdf2
    for index, row in gdf.iterrows():
        # Get the value of the "NOM" column for the current row in gdf2
        name_gdf = row[column_name]
        highest_similarity_ratio = 0
        best_matching_name = None
        # Iterate through unique names in gdf1
        for unique_name in unique_names:
            # Calculate similarity ratio between names in gdf2 and gdf1
            similarity_ratio = SequenceMatcher(None, unique_name, name_gdf).ratio()
            # Update best matching name if similarity ratio is higher
            if similarity_ratio > highest_similarity_ratio:
                highest_similarity_ratio = similarity_ratio
                best_matching_name = unique_name

        if highest_similarity_ratio >= similarity_threshold:
            # confirmation = input(f"Similarity found: '{name_gdf2}' -> '{name_gdf1}'Is this okay? (y/n): ").strip().lower()
            # if confirmation == "y":
            # print(f"{highest_similarity_ratio} for {name_gdf1} to {best_matching_name}")
            gdf.at[index, column_name] = best_matching_name

    return gdf

def create_identifier_dictionary(list):
    """
    Creates a dictionary out of a list appending a new id to each one of them.

    Parameters:
    - list (list): list of strings.

    Returns:
    - value_to_text_dict. dictionary o value: text.
    """
    value_to_text_dict = {value: index + 1 for index, value in enumerate(sorted(list))}
    return value_to_text_dict

def rasterize_geodataframe_by_column(gdf, column_name, value_to_index, resolution, output_path):
    """
    Rasterizes a geodataframe based on the column field.

    Parameters:
    - gdf (GeoDataFrame): GeoDataFrame to be rasterized.
    - column_name (str): String of the column name.
    - resolution (int): resolution of the raster.
    - output_path (str): output of the raster file.

    Returns:
    - None. Rasterizes the geodataframe.
    """
    # It takes the column reference and act depending if they are intergers o strings.
    if gdf[column_name].dtype == object:
        print(f"The column '{column_name}' contains strings.")
        # Add a new column to the GeoDataFrame containing the unique identifiers
        
        gdf['raster_value'] = gdf[column_name].map(value_to_index)
    else:
        print(f"The column '{column_name}' does not contain strings.")
        # Get unique values/strings from the specified column, they are always sorted.
        unique_values = sorted(gdf[column_name].unique())
        # Create a dictionary one to one
        value_to_index = {value: value for value in unique_values}

    
    # Get the bounds of the GeoDataFrame
    xmin, ymin, xmax, ymax = gdf.total_bounds
    # Calculate the number of pixels in x and y directions
    cols = int((xmax - xmin) / resolution)
    rows = int((ymax - ymin) / resolution)
    # Create a transform for the raster
    transform = from_origin(xmin, ymax, resolution, resolution)

    # Create an empty array to hold the rasterized values
    rasterized_array = np.zeros((rows, cols), dtype=np.uint8) # if bigger, change the dtype. This is crucial.

    nodata_value = 255
    # Rasterize each unique value separately
    for text_value, value in value_to_index.items():
        # print(f"Finished with {text_value}")
        mask = gdf['raster_value'] == value
        shapes = gdf.loc[mask, 'geometry']
        temp_raster = features.rasterize(
            shapes=shapes,
            out_shape=(rows, cols),
            # fill=nodata_value, # this covers areas that are not covered by geometries
            transform=transform,
            all_touched=True, # Esto asegura que si toca la linea del poligono, se genera el pixel
            default_value=value,
            dtype=np.uint8, # must be equal to the zeros
        )
        rasterized_array = np.maximum(rasterized_array, temp_raster)

    crs = gdf.crs

    # Define the metadata for the raster
    profile = {
        'driver': 'GTiff',
        'height': rows,
        'width': cols,
        'count': 1,
        'dtype': rasterio.uint8,
        'crs': crs, #CRS.from_epsg(32628),
        'transform': transform,
        'nodata': nodata_value,  # Set the nodata value in the profile metadata
        'compress': 'deflate',  # Compression method
        'tiled': True,  # Enable tiling
        'legend': {str(key): value for key, value in value_to_index.items()}
    }

    # Write the raster array to a GeoTIFF file
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(rasterized_array, 1)

        # Set nodata values in the raster
        rasterized_array[rasterized_array == 0] = nodata_value
        dst.write(rasterized_array, 1)
    

def check_same_dimensions(raster_files):
    """
    Check the if the dimensions of all the input rasters have the same dimensions.

    Parameters:
    - raster_files (list): List of raster files.

    Returns:
    - None. Validates the dimensions of all the raster files.
    """
    # Open the first raster file in the list
    with rasterio.open(raster_files[0]) as src:
        first_shape = src.shape
    
    # Iterate over the remaining raster files
    for file_path in raster_files[1:]:
        with rasterio.open(file_path) as src:
            # Check if dimensions are the same
            if src.shape != first_shape:
                print(f"Dimensions of {file_path} do not match the first raster file")
                return False
    
    # If all dimensions match, return True
    print("All raster files have the same dimensions")


def read_csv_in_pairs(csv_file):
    """
    Reads a two column csv and transforms it into a pair value list.

    Parameters:
    - csv_file (str): path of the csv file.

    Returns:
    - rule_values_list: list of unique value pairs.
    """
    rule_values_list = []
    with open(csv_file, 'r') as file:
        # next(file)  # Skip the header row
        for line in file:
            # Split the line into two values based on spaces, and convert them to floats
            rule_value_1, rule_value_2 = map(float, line.split(","))
            rule_values_list.append((rule_value_1, rule_value_2))
    return rule_values_list

def read_csv_columns_and_data(csv_file):
    """
    Reads a two column csv and transforms it into a pair value list.

    Parameters:
    - csv_file (str): path of the csv file.

    Returns:
    - columns: list of columns.
    - rows: list of rows.
    """

    rows = ()
    with open(csv_file, 'r') as file:
        for line in file:
            if len(columns) == 0:
                columns = (line)
            values = line.strip().split(",")
            rows.append(values)
    return columns, rows


def generate_random_pairs(paths_list):
    """
    Generates a list of random unique pairs, from the input list.

    Parameters:
    - paths_list (list): list of paths.

    Returns:
    - all_pairs: list of unique value pairs.
    """
    # Generate all possible pairs from the raster paths
    all_pairs = list(itertools.combinations(paths_list, 2))
    # Randomly shuffle the list of pairs
    random.shuffle(all_pairs)
    return all_pairs


def compare_rasters(raster_pair, output, rule_table_path):
    """
    Generates a list of random unique pairs, from the input list.

    Parameters:
    - raster_pair (list): list of two elements.
    - output (path): path of the output file.
    - rule_table_path (list): is a list of pairs.

    Returns:
    - output_loc (str): path out the ourput file. 
    - comparison_df: dataframe related to the data.
    """
    # Open raster files
    ds1 = gdal.Open(raster_pair[0])
    ds2 = gdal.Open(raster_pair[1])

    if not ds1 or not ds2:
        print("Error: Unable to open raster files.")
        return
    
    # Check if both rasters have the same height and width
    if ds1.RasterXSize != ds2.RasterXSize or ds1.RasterYSize != ds2.RasterYSize:
        print(f"Error: Rasters have different dimensions.")
    
    # Get the first raster information
    width = ds1.RasterXSize
    height = ds1.RasterYSize
    geotransform = ds1.GetGeoTransform()
    projection = ds1.GetProjection()
    
    # Read rule table and create a list of pairs with the info
    rule_values_list = read_csv_in_pairs(rule_table_path)

    # Create output raster
    driver = gdal.GetDriverByName("GTiff")
    year1 = os.path.basename(raster_pair[0]).split('_')[1]
    year2 = os.path.basename(raster_pair[1]).split('_')[1]
    output_filename = f"occsol_{year1}_{year2}_anat_illogical_transitions.tif" #Customize
    output_loc = os.path.join(output, output_filename)

    output_ds = driver.Create(output_loc, width, height, 1, gdal.GDT_Int16, options= ['COMPRESS=DEFLATE', 'TILED=YES']) # GDT_Int32
    output_ds.GetRasterBand(1).SetNoDataValue(0)
    output_ds.SetGeoTransform(geotransform)
    output_ds.SetProjection(projection)

    output_array = np.zeros((height, width), dtype=np.int16) # int16

    unique_value_dict = {} # pairs : unique_value


    # Loop through each pixel and compare values
    block_size = 256  # Adjust the block size as needed
    for y in range(0, height, block_size):
        for x in range(0, width, block_size):
            print(f"Comparing pixels at rows/columns ({x},{y}) from ({width}, {height})", end='\r')
            block_width = min(block_size, width - x)
            block_height = min(block_size, height - y)

            block1 = ds1.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
            block2 = ds2.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
    
            # Check if the pair of values matches any rule pair
            correct_value = 0
            
            rule_values_list = set(rule_values_list) # To speed up the process, don't use list
            for i in range(block_height):
                for j in range(block_width):
                    value1 = block1[i, j]
                    value2 = block2[i, j]

                    if (value1, value2) in rule_values_list:
                        unique_value = unique_value_dict.setdefault((value1, value2), len(unique_value_dict) + 1)
                        output_array[y + i, x + j] = unique_value

                    else:
                        output_array[y + i, x + j] = correct_value

    # Write the output
    output_ds.GetRasterBand(1).WriteArray(output_array)

    # Close datasets
    ds1 = None
    ds2 = None
    output_ds = None

    # Create the dataframe
    unique_values = list(unique_value_dict.values())
    value_pairs = list(unique_value_dict.keys())

    comparison_df = pd.DataFrame({
        'univalue': unique_values,
        f'Val1_{year1}': [pair[0] for pair in value_pairs],
        f'Val2_{year2}': [pair[1] for pair in value_pairs]
                                })
    
    # Lets export the csv just in case
    comparison_df.to_csv(os.path.join(output, f"illogical_table_{year1}_{year2}.csv"), index=False)

    return output_loc, comparison_df


def vectorize_raster(raster_path, output_path, df):
    """
    Vectorizes an input raster.

    Parameters:
    - raster_path (str): path of the input raster.
    - output_path (path): path of the output file.
    - df (dataframe): dataframe related to the input raster.

    Returns:
    - None. It produces the vector file. 
    """
    # Open the raster file
    with rasterio.open(raster_path) as src:
        # Read raster data as numpy array
        data = src.read(1)
        # Get affine transform of the raster
        transform = src.transform

    # Vectorize the raster data
    raster_shapes = features.shapes(data, transform=transform)

    # Convert the vectorized shapes into a GeoDataFrame
    gdf = gpd.GeoDataFrame.from_features(
        [
            {"geometry": geo_shape, "properties": {"univalue": value}}
            for geo_shape, value in raster_shapes
        ],
        crs=src.crs
    )

    # Merge the dataframe with the GeoDataFrame
    merged_gdf = gdf.merge(df, on='univalue')
    # Dissolve based on a column value
    dissolved_gdf = merged_gdf.dissolve(by='univalue')

    # Save the merged GeoDataFrame as a new shapefile
    filename = os.path.join(output_path, os.path.basename(raster_path).split('.')[0])
    dissolved_gdf.to_file(f'{filename}.shp', driver='ESRI Shapefile')
    return

def add_name_columns_to_dataframe(df, column_name, names_dictionary):
    """
    Adds the input name list into the  an input raster.

    Parameters:
    - df (dataframe): dataframe related to the input raster.
    - column_name (str): name of the column that is appeended to the df.
    - names_dictionary (dict): dict of the names that we want to append.

    Returns:
    - None. It produces the vector file. 
    """
    reversed_dict = {v: k for k, v in names_dictionary.items()} # Reverse it to match the values / value: text
    count = 1
    for column in df.columns:
        if column.startswith('Value'):
            df[column_name + str(count)] = df[f"Value{count}"].map(reversed_dict)
            count += 1
    return df

def consecutive_pairs(list):
    """
    Creates a consecutive pairs list.

    Parameters:
    - list (list): list of files.

    Returns:
    - pairs (list): list of pairs in order. 
    """
    pairs = []
    for i in range(len(list) - 1):
        pairs.append((list[i], list[i + 1]))
    return pairs

In [4]:
"""Specify all the inputs"""

path = r"Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat"
rule_table_path = r"Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat\illogical_transitions.csv"
output_path = path + r"\output_files"

create_folder_if_not_exists(output_path)

# The input will always be text + id
names_list = ["Carrière Mine Infrastructure", "Cours d'eau", "Culture irriguée", "Culture maraîchère", "Culture pluviale", "Dune", "Forêt", "Forêt galerie", "Lac", "Localité", "Mangrove", "Mare", "Plaine inondable", "Plantation forestière", "Prairie aquatique", "Savane", "Savane arbustive", "Sol nu", "Steppe", "Tanne", "Vasière"]

# Here we set a value to each string, since in this case it does not have it
names_dictionary = create_identifier_dictionary(names_list)

vector_file_list = get_vector_file_list(path)

# Specify the column to rasterize by
column_name = 'NOM'
# Define the resolution of your raster.
resolution = 50  # in meters

Folder already exists at: Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat\output_files


In [ ]:
"""Transform the vector files and convert them into rasters"""
for file in vector_file_list[0:]:
    output_path_file = os.path.join(output_path, os.path.basename(file).replace(".shp", ".tif"))
    print(output_path_file)
    gdf = gpd.read_file(file)
    gdf = update_names_based_on_similarity(names_list, gdf, column_name, similarity_threshold=0.5)
    print("Updated text values based on similarity")
    if names_dictionary:
        print("The user specified the names dictionary")
        rasterize_geodataframe_by_column(gdf, column_name, names_dictionary, resolution, output_path_file)
    else:
        names_dictionary=None
        rasterize_geodataframe_by_column(gdf, column_name, names_dictionary, resolution, output_path_file)

In [28]:
"""Create the illogical transitions 1 to 1 and vectorize it"""

illogical_path = path + r"\illogical_files"
create_folder_if_not_exists(illogical_path)

rule_values_list = read_csv_in_pairs(rule_table_path)
raster_list = get_raster_file_list(output_path)
all_pairs_path_list = generate_random_pairs(raster_list)

illogical_info = {}

for raster_pair in all_pairs_path_list[:]:
    illogical_raster_path, illogical_df = compare_rasters(raster_pair, illogical_path, rule_table_path)
    illogical_df = add_name_columns_to_dataframe(illogical_df, column_name, names_dictionary)
    
    # Store the illogical raster path and the corresponding df.
    illogical_info[illogical_raster_path] = illogical_df

    # vectorize_raster(illogical_raster_path, output_path, illogical_df)


Folder already exists at: Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat\illogical_files


In [7]:
"""Build the year finder"""
illogical_path = path + r"\illogical_files"

# Search for the input rasters
raster_list = get_raster_file_list(output_path)
years_list = [re.findall(r'\d+', os.path.basename(file))[0] for file in raster_list] # We expect to only have one year
year_pairs = [f"{years_list[i]}_{years_list[i + 1]}" for i in range(len(years_list) - 1)]

illogical_rasters = get_raster_file_list(illogical_path)
illogical_csvs = get_csv_file_list(illogical_path)

In [18]:
for index in range(len(year_pairs) - 1):   
    raster_file_1 = [raster for raster in illogical_rasters if year_pairs[index] in raster][0]
    raster_file_2 = [raster for raster in illogical_rasters if year_pairs[index + 1] in raster][0]

    csv_1 = [raster for raster in illogical_csvs if year_pairs[index] in raster][0]
    csv_2 = [raster for raster in illogical_csvs if year_pairs[index + 1] in raster][0]

    df_1 = pd.read_csv(csv_1)
    df_2 = pd.read_csv(csv_2)

In [27]:

columns = set()
data = set()
with open(csv_1, 'r') as file:
    header = next(file).strip().split(",")  # Extract header names
    columns.update(header)  # Update columns set with header names
    for line in file:
        values = line.strip().split(",")
        data.add(tuple(values))  # Add row as a tuple to the data set



In [19]:
rows, columns = read_csv_columns_and_data(csv_1)

In [49]:
combination_df = pd.DataFrame()

row_1 = df_1.loc[df_1["univalue"] == value1].drop(columns=["univalue"])
row_2 = df_2.loc[df_2["univalue"] == value2].drop(columns=["univalue"])

# Combine the rows into combination_df
combination_df = pd.concat([row_1.reset_index(drop=True), row_2.reset_index(drop=True)], axis=1)

# Assign a unique value to "univalue" column
combination_df["univalue"] = len(combination_df)



In [36]:
# We are going to do a recursive function.
for index in range(len(year_pairs) - 1):   
    raster_file_1 = [raster for raster in illogical_rasters if year_pairs[index] in raster][0]
    raster_file_2 = [raster for raster in illogical_rasters if year_pairs[index + 1] in raster][0]

    csv_1 = [raster for raster in illogical_csvs if year_pairs[index] in raster][0]
    csv_2 = [raster for raster in illogical_csvs if year_pairs[index + 1] in raster][0]

    n = len(year_pairs) - 1

    ds1 = gdal.Open(raster_file_1)
    ds2 = gdal.Open(raster_file_2)

    if not ds1 or not ds2:
        print("Error: Unable to open raster files.")
        # return
    
    # Check if both rasters have the same height and width
    if ds1.RasterXSize != ds2.RasterXSize or ds1.RasterYSize != ds2.RasterYSize:
        print(f"Error: Rasters have different dimensions.")
    
    # Get the first raster information
    width = ds1.RasterXSize
    height = ds1.RasterYSize
    geotransform = ds1.GetGeoTransform()
    projection = ds1.GetProjection()
    
    # Read rule table and create a list of pairs with the info
    df_1 = pd.read_csv(csv_1)
    df_2 = pd.read_csv(csv_2)

    # Create output raster
    driver = gdal.GetDriverByName("GTiff")
    year1 = os.path.basename(year_pairs[0]).split('_')[1]
    year2 = os.path.basename(year_pairs[1]).split('_')[1]
    output_filename = f"occsol_{n}_anat_illogical_transitions.tif" #Customize
    output_loc = os.path.join(output_path, output_filename)

    output_ds = driver.Create(output_loc, width, height, 1, gdal.GDT_Int16, options= ['COMPRESS=DEFLATE', 'TILED=YES']) # GDT_Int32
    output_ds.GetRasterBand(1).SetNoDataValue(0) # Set nodata value
    output_ds.SetGeoTransform(geotransform)
    output_ds.SetProjection(projection)

    output_array = np.zeros((height, width), dtype=np.int16) # int16

    combination_df = pd.DataFrame()

    unique_value_dict = {} # pairs : unique_value
    # Loop through each pixel and compare values
    block_size = 256  # Adjust the block size as needed
    for y in range(0, height, block_size):
        for x in range(0, width, block_size):
            print(f"Comparing pixels at rows/columns ({x},{y}) from ({width}, {height})") # , end='\r'
            block_width = min(block_size, width - x)
            block_height = min(block_size, height - y)

            block1 = ds1.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
            block2 = ds2.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
    
            # Check if the pair of values matches any rule pair
            correct_value = 0
            
            # rule_values_list = set(rule_values_list)

            for i in range(block_height):
                for j in range(block_width):
                    value1 = block1[i, j]
                    value2 = block2[i, j]

                    if value1 != 0 or value2 != 0:
                        print(f"{value1}_{value2}passed if")
                        # value can be 0, so we have to fix that
                        # print(row_1)
                        # row_1 = df_1.loc[df_1["univalue"] == value1].drop(columns=["univalue"])
                        # row_2 = df_2.loc[df_2["univalue"] == value2].drop(columns=["univalue"])

                        # # Combine the rows into combination_df
                        # combination_df = pd.concat([row_1.reset_index(drop=True), row_2.reset_index(drop=True)], axis=1)

                        # # Serialize the combined DataFrame to a JSON string
                        # combination_json = combination_df.to_json()

                        # unique_value = unique_value_dict.setdefault(combination_json, len(unique_value_dict) + 1)


                        output_array[y + i, x + j] = 1
                    else:
                        output_array[y + i, x + j] = correct_value

    output_ds.GetRasterBand(1).WriteArray(output_array)
    # Close datasets
    ds1 = None
    ds2 = None
    output_ds = None


    # return output_loc, comparison_df

Comparing pixels at rows/columns (0,0) from (13385, 9670)
Comparing pixels at rows/columns (256,0) from (13385, 9670)
Comparing pixels at rows/columns (512,0) from (13385, 9670)
Comparing pixels at rows/columns (768,0) from (13385, 9670)
Comparing pixels at rows/columns (1024,0) from (13385, 9670)
Comparing pixels at rows/columns (1280,0) from (13385, 9670)
Comparing pixels at rows/columns (1536,0) from (13385, 9670)
Comparing pixels at rows/columns (1792,0) from (13385, 9670)
Comparing pixels at rows/columns (2048,0) from (13385, 9670)


KeyboardInterrupt: 

In [16]:
combination_json

'{"Val1_2010":{"0":19},"Val2_2015":{"0":13},"Val1_2015":{"0":null},"Val2_2020":{"0":null}}'

In [ ]:
def compare_rasters(raster_pair, output, rule_table_path):
    """
    Generates a list of random unique pairs, from the input list.

    Parameters:
    - raster_pair (list): list of two elements.
    - output (path): path of the output file.
    - rule_table_path (list): is a list of pairs.

    Returns:
    - output_loc (str): path out the ourput file. 
    - comparison_df: dataframe related to the data.
    """
    # Open raster files
    ds1 = gdal.Open(raster_pair[0])
    ds2 = gdal.Open(raster_pair[1])

    if not ds1 or not ds2:
        print("Error: Unable to open raster files.")
        return
    
    # Check if both rasters have the same height and width
    if ds1.RasterXSize != ds2.RasterXSize or ds1.RasterYSize != ds2.RasterYSize:
        print(f"Error: Rasters have different dimensions.")
    
    # Get the first raster information
    width = ds1.RasterXSize
    height = ds1.RasterYSize
    geotransform = ds1.GetGeoTransform()
    projection = ds1.GetProjection()
    
    # Read rule table and create a list of pairs with the info
    rule_values_list = read_csv_in_pairs(rule_table_path)

    # Create output raster
    driver = gdal.GetDriverByName("GTiff")
    year1 = os.path.basename(raster_pair[0]).split('_')[1]
    year2 = os.path.basename(raster_pair[1]).split('_')[1]
    output_filename = f"occsol_{year1}_{year2}_anat_illogical_transitions.tif" #Customize
    output_loc = os.path.join(output, output_filename)

    output_ds = driver.Create(output_loc, width, height, 1, gdal.GDT_Int16, options= ['COMPRESS=DEFLATE', 'TILED=YES']) # GDT_Int32
    output_ds.GetRasterBand(1).SetNoDataValue(0) # Set nodata value
    output_ds.SetGeoTransform(geotransform)
    output_ds.SetProjection(projection)

    output_array = np.zeros((height, width), dtype=np.int16) # int16

    unique_value_dict = {} # pairs : unique_value
    # Loop through each pixel and compare values
    block_size = 256  # Adjust the block size as needed
    for y in range(0, height, block_size):
        for x in range(0, width, block_size):
            print(f"Comparing pixels at rows/columns ({x},{y}) from ({width}, {height})", end='\r')
            block_width = min(block_size, width - x)
            block_height = min(block_size, height - y)

            block1 = ds1.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
            block2 = ds2.GetRasterBand(1).ReadAsArray(x, y, block_width, block_height)
    
            # Check if the pair of values matches any rule pair
            correct_value = 0
            
            rule_values_list = set(rule_values_list) # To speed up the process, don't use list
            for i in range(block_height):
                for j in range(block_width):
                    value1 = block1[i, j]
                    value2 = block2[i, j]

                    if (value1, value2) in rule_values_list:
                        unique_value = unique_value_dict.setdefault((value1, value2), len(unique_value_dict) + 1)
                        output_array[y + i, x + j] = unique_value
                    else:
                        output_array[y + i, x + j] = correct_value

    output_ds.GetRasterBand(1).WriteArray(output_array)
    # Close datasets
    ds1 = None
    ds2 = None
    output_ds = None

    # Create the dataframe
    unique_values = list(unique_value_dict.values())
    value_pairs = list(unique_value_dict.keys())

    comparison_df = pd.DataFrame({
        'univalue': unique_values,
        'Value1': [pair[0] for pair in value_pairs],
        'Value2': [pair[1] for pair in value_pairs]
                                })
    
    return output_loc, comparison_df

In [ ]:
"""IN PROGRESS Create illogical transitions altogether and vectorize it"""
rule_values_list = read_csv(rule_table_path)
all_pairs_path_list = consecutive_pairs(raster_list)

# Get the first raster pair
for raster_pair in all_pairs_path_list[0:]:
    illogical_raster_path, illogical_df = compare_rasters(raster_pair, output_path, rule_table_path)

# Union the create vector files.

In [ ]:
def iterate_rasters(input_folder): # Esto ya no sirve para mucho pero no quiero borrarlo
    """Provides two lists of files that go in pairs"""
    # Get a list of raster files in the input folder
    raster_files = [filename for filename in os.listdir(input_folder) if filename.endswith((".tif", ".tiff"))]

    # Create two empty lists
    first_file_list = []
    second_file_list = []

    # Iterate through pairs of current and next files
    for i in range(len(raster_files) - 1):
        current_file = os.path.join(input_folder, raster_files[i])
        next_file = os.path.join(input_folder, raster_files[i + 1])
        # If the second file is empty, finish
        if not next_file:
            print("Finished getting both values")
            continue
        else:
            first_file_list.append(current_file)
            second_file_list.append(next_file)

    return first_file_list, second_file_list